In [2]:
import json
from typing import List, Optional
from google import genai
from pydantic import BaseModel, Field
import pandas as pd
import numpy as np
import pickle
import bm25s
from sentence_transformers import SentenceTransformer
import faiss
import re

## Query

In [3]:
query = "Fast-growing fintech companies competing with traditional banks in Europe."

In [4]:
query = "Public software companies with more than 1,000 employees."

In [87]:
query = "Logistic companies in Romania"

In [80]:
query = "Food and beverage manufacturers in France"

In [90]:
query = "Companies that could supply packaging materials for a direct-to-consumer cosmetics brand"

In [103]:
query = "Construction companies in the United States with revenue over $50 million"

In [108]:
query = "Pharmaceutical companies in Switzerland"

In [111]:
query = "B2B SaaS companies providing HR solutions in Europe"

In [122]:
query = "Clean energy startups founded after 2018 with fewer than 200 employees"

In [125]:
query = "E-commerce companies using Shopify or similar platforms"

In [145]:
query = "Renewable energy equipment manufacturers in Scandinavia"

In [142]:
query = "Companies that manufacture or supply critical components for electric vehicle battery production"

# Pasul 1

In [ ]:
client = genai.Client(api_key="")

In [5]:
SYSTEM_PROMPT = """
You are an expert query parser for a company search engine.

Your task is to convert a user query into a structured search plan.

The company database may contain fields such as:
- year_founded
- address
- employee_count
- revenue
- is_public
- primary_naics
- secondary_naics
- description
- business_model
- target_markets
- core_offerings

Rules:
- Extract only constraints explicitly stated or strongly implied by the query.
- Do not invent unsupported facts, thresholds, geographies, or attributes.
- For numeric constraints, use min/max style fields when appropriate.
- Generate business-language terms likely to appear in description, NAICS labels, business_model, target_markets, and core_offerings.
- Generate offering-related terms when the query implies products or services.
- Generate target-market terms when the query implies customer segments or served industries.
- Use exclude_terms only when the query clearly implies exclusions.
- Generate a short target_description suitable for semantic retrieval.
- Preserve ambiguity when the query is vague.

- Extract geographic mentions into top-level QueryPlan geography fields:
  - put explicit or strongly implied countries into countries
  - put explicit geographic regions, continents, economic blocs, or political groupings into regions

- hard_filters.country_codes is the executable geography filter for search-time filtering.

- When the query specifies a country:
  - add the country name to countries
  - add its ISO 3166-1 alpha-2 code to hard_filters.country_codes

- When the query specifies a city or locality:
  - infer its country only if the mapping is clear and unambiguous in context
  - if so, add the country name to countries
  - and add its ISO 3166-1 alpha-2 code to hard_filters.country_codes
  - otherwise, do not invent a country code

- When the query specifies a geographic region, continent, economic bloc, or political grouping
  (for example Europe, Scandinavia, Nordics, EU, NATO, BRICS, DACH):
  - preserve the original mention in regions
  - expand it into the corresponding ISO 3166-1 alpha-2 country codes
  - place those codes in hard_filters.country_codes

- Only include geographic filters when the geography is explicit or strongly implied.
- Do not invent unsupported geographic constraints.
- If a geographic mention is ambiguous, preserve the ambiguity and only add country codes when the mapping is reliable.
- hard_filters.allowed_country_codes must contain unique uppercase ISO 3166-1 alpha-2 codes.

- Return valid JSON only.
"""

In [6]:
class HardFilters(BaseModel):
    country_codes: List[str] = []
    year_founded_min: Optional[int] = None
    year_founded_max: Optional[int] = None
    revenue_min: Optional[float] = None
    revenue_max: Optional[float] = None
    employee_count_min: Optional[int] = None
    employee_count_max: Optional[int] = None
    is_public: Optional[bool] = None

class QueryPlan(BaseModel):
    hard_filters: HardFilters
    countries: List[str] = Field(default_factory=list)
    regions: List[str] = Field(default_factory=list)
    industry_terms: List[str] = Field(default_factory=list)
    naics_terms: List[str] = Field(default_factory=list)
    business_model_terms: List[str] = Field(default_factory=list)
    offering_terms: List[str] = Field(default_factory=list)
    target_market_terms: List[str] = Field(default_factory=list)
    canonical_terms: List[str] = Field(default_factory=list)
    exclude_terms: List[str] = Field(default_factory=list)
    target_description: str

In [7]:
def parse_query(user_query: str) -> QueryPlan:
    prompt = f"""
{SYSTEM_PROMPT}

Parse this user query into the schema.

User query:
{user_query}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": QueryPlan,
            "temperature": 0,
        },
    )

    return response.parsed

In [8]:
plan = parse_query(query)
print(plan.model_dump_json(indent=2))

{
  "hard_filters": {
    "country_codes": [
      "AT",
      "BE",
      "BG",
      "HR",
      "CY",
      "CZ",
      "DK",
      "EE",
      "FI",
      "FR",
      "DE",
      "GR",
      "HU",
      "IE",
      "IT",
      "LV",
      "LT",
      "LU",
      "MT",
      "NL",
      "PL",
      "PT",
      "RO",
      "SK",
      "SI",
      "ES",
      "SE",
      "IS",
      "LI",
      "NO",
      "CH",
      "GB"
    ],
    "year_founded_min": null,
    "year_founded_max": null,
    "revenue_min": null,
    "revenue_max": null,
    "employee_count_min": null,
    "employee_count_max": null,
    "is_public": null
  },
  "countries": [
    "Austria",
    "Belgium",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "Czech Republic",
    "Denmark",
    "Estonia",
    "Finland",
    "France",
    "Germany",
    "Greece",
    "Hungary",
    "Ireland",
    "Italy",
    "Latvia",
    "Lithuania",
    "Luxembourg",
    "Malta",
    "Netherlands",
    "Poland",
    "Portugal",
    "Rom

In [9]:
# SALVARE
with open("plan.json", "w", encoding="utf-8") as f:
    json.dump(plan.model_dump(), f, indent=2, ensure_ascii=False)

In [10]:
queryFile = "plan.json"

In [11]:
# CITIRE (data viitoare, fără LLM)
with open(queryFile, "r", encoding="utf-8") as f:
    plan_data = json.load(f)
plan = QueryPlan.model_validate(plan_data)

print(plan.model_dump_json(indent=2))

{
  "hard_filters": {
    "country_codes": [
      "AT",
      "BE",
      "BG",
      "HR",
      "CY",
      "CZ",
      "DK",
      "EE",
      "FI",
      "FR",
      "DE",
      "GR",
      "HU",
      "IE",
      "IT",
      "LV",
      "LT",
      "LU",
      "MT",
      "NL",
      "PL",
      "PT",
      "RO",
      "SK",
      "SI",
      "ES",
      "SE",
      "IS",
      "LI",
      "NO",
      "CH",
      "GB"
    ],
    "year_founded_min": null,
    "year_founded_max": null,
    "revenue_min": null,
    "revenue_max": null,
    "employee_count_min": null,
    "employee_count_max": null,
    "is_public": null
  },
  "countries": [
    "Austria",
    "Belgium",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "Czech Republic",
    "Denmark",
    "Estonia",
    "Finland",
    "France",
    "Germany",
    "Greece",
    "Hungary",
    "Ireland",
    "Italy",
    "Latvia",
    "Lithuania",
    "Luxembourg",
    "Malta",
    "Netherlands",
    "Poland",
    "Portugal",
    "Rom

# Pasul 2

### Construirea query ului pentru BM25

In [12]:
def deduplicate_preserve_order(items):
    seen = set()
    result = []
    for item in items:
        if not item:
            continue
        key = item.strip().lower()
        if key not in seen:
            seen.add(key)
            result.append(item.strip())
    return result

In [13]:
def build_bm25_query(plan):
    terms = []
    terms.extend(plan.industry_terms)
    terms.extend(plan.naics_terms)
    terms.extend(plan.business_model_terms)
    terms.extend(plan.offering_terms)
    terms.extend(plan.target_market_terms)
    terms.extend(plan.canonical_terms)
    terms.extend(plan.countries)
    terms.extend(plan.regions)

    terms = deduplicate_preserve_order(terms)
    return " ".join(terms)

In [14]:
bm25_query = build_bm25_query(plan)
print(bm25_query)

fintech financial technology banking financial services Commercial Banking Investment Banking and Securities Dealing Credit Card Issuing Other Nondepository Credit Intermediation Depository Credit Intermediation Nondepository Credit Intermediation challenger bank digital bank alternative finance disruptive finance neobank payment solutions lending platforms digital banking online payments mobile banking investment platforms payment processing financial software wealth management technology insurtech consumers small businesses enterprises retail banking customers corporate banking customers Austria Belgium Bulgaria Croatia Cyprus Czech Republic Denmark Estonia Finland France Germany Greece Hungary Ireland Italy Latvia Lithuania Luxembourg Malta Netherlands Poland Portugal Romania Slovakia Slovenia Spain Sweden Iceland Liechtenstein Norway Switzerland United Kingdom Europe


### Aplicarea algoritmului BM25

In [15]:
df_proc = pd.read_pickle("artifacts/companies_processed.pkl")

with open("artifacts/bm25_index.pkl", "rb") as f:
    bm25 = pickle.load(f)

In [16]:
query_tokens = bm25s.tokenize([bm25_query])

bm25_results, bm25_scores = bm25.retrieve(query_tokens, k=10)

bm25_ids = bm25_results[0]
bm25_scores = bm25_scores[0]

In [17]:
bm25_results_df = df_proc.iloc[bm25_ids].copy()
bm25_results_df["bm25_score"] = bm25_scores

print(
    bm25_results_df[
        ["bm25_score", "operational_name", "is_public", "employee_count", "year_founded", "description"]
    ].to_string()
)

     bm25_score              operational_name  is_public  employee_count  year_founded                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          description
57    36.038319             

### Construirea query ului pentru SentenceTransformer

In [20]:
def build_embedding_query(plan):
    return plan.target_description.strip()

In [21]:
embedding_query = build_embedding_query(plan)
print(embedding_query)

Fast-growing fintech companies competing with traditional banks in Europe.


### Aplicarea algoritmului Semantic Retrieval

In [22]:
with open("artifacts/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

index = faiss.read_index("artifacts/company_faiss.index")
print(metadata)

{'embedding_model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'num_companies': 457, 'embedding_dim': 384}


In [23]:
embedder = SentenceTransformer(metadata["embedding_model_name"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4026.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
query_embedding = embedder.encode(
    [embedding_query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

In [25]:
semantic_scores, semantic_ids = index.search(query_embedding, 10)

sem_ids = semantic_ids[0]
sem_scores = semantic_scores[0]

In [26]:
semantic_results_df = df_proc.iloc[sem_ids].copy()
semantic_results_df["semantic_score"] = sem_scores

print(
    semantic_results_df[
        ["semantic_score", "operational_name", "is_public", "employee_count", "year_founded", "description"]
    ].to_string()
)

     semantic_score              operational_name  is_public  employee_count  year_founded                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          description
62         0.418768     

### Scor final BM25 + FAISS

In [27]:
def min_max_normalize(score_dict):
    if not score_dict:
        return {}

    values = np.array(list(score_dict.values()), dtype=float)
    min_v = values.min()
    max_v = values.max()

    if max_v - min_v < 1e-12:
        return {k: 1.0 for k in score_dict}

    return {k: (v - min_v) / (max_v - min_v) for k, v in score_dict.items()}

In [28]:
bm25_score_dict = {
    int(doc_id): float(score)
    for doc_id, score in zip(bm25_ids, bm25_scores)
}

semantic_score_dict = {
    int(doc_id): float(score)
    for doc_id, score in zip(sem_ids, sem_scores)
}

In [29]:
bm25_norm = min_max_normalize(bm25_score_dict)
semantic_norm = min_max_normalize(semantic_score_dict)

In [30]:
print(bm25_norm)
print(semantic_norm)

{57: np.float64(1.0), 52: np.float64(0.8632895100920995), 53: np.float64(0.32725483666254435), 55: np.float64(0.32024522556315166), 59: np.float64(0.27411227550376926), 62: np.float64(0.1874626759374778), 61: np.float64(0.11484774144342176), 58: np.float64(0.10602023161766642), 238: np.float64(0.10234519109829171), 273: np.float64(0.0)}
{62: np.float64(1.0), 61: np.float64(0.9390180516770491), 53: np.float64(0.7710983290882868), 54: np.float64(0.47263766992972656), 56: np.float64(0.2988672630189953), 23: np.float64(0.28693957713198515), 57: np.float64(0.2471379499493663), 59: np.float64(0.20642625893761316), 55: np.float64(0.16093936999416947), 424: np.float64(0.0)}


In [31]:
all_doc_ids = set(bm25_norm.keys()) | set(semantic_norm.keys())

In [32]:
all_doc_ids

{23, 52, 53, 54, 55, 56, 57, 58, 59, 61, 62, 238, 273, 424}

In [33]:
rows = []

for doc_id in all_doc_ids:
    bm25_score = bm25_norm.get(doc_id, 0.0)
    semantic_score = semantic_norm.get(doc_id, 0.0)

    final_score = 0.5 * bm25_score + 0.5 * semantic_score

    rows.append({
        "doc_id": doc_id,
        "bm25_score_norm": bm25_score,
        "semantic_score_norm": semantic_score,
        "retrieval_score": final_score,
    })

In [34]:
fusion_df = pd.DataFrame(rows).sort_values(
    "retrieval_score", ascending=False
).reset_index(drop=True)

In [35]:
top_ids = fusion_df["doc_id"].head(10).tolist()

final_results = df_proc.iloc[top_ids].copy()
final_results["bm25_score_norm"] = fusion_df["bm25_score_norm"].head(10).values
final_results["semantic_score_norm"] = fusion_df["semantic_score_norm"].head(10).values
final_results["retrieval_score"] = fusion_df["retrieval_score"].head(10).values

In [36]:
print(
    final_results[
        ["retrieval_score", "bm25_score_norm", "semantic_score_norm", "operational_name", "is_public", "employee_count", "year_founded", "description"]
    ].to_string()
)

    retrieval_score  bm25_score_norm  semantic_score_norm              operational_name  is_public  employee_count  year_founded                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

# Pasul 3

In [37]:
import math
import pandas as pd


def is_missing(value) -> bool:
    if value is None:
        return True
    if pd.isna(value):
        return True
    return False


def passes_min(value, min_value) -> bool:
    if min_value is None:
        return True
    if is_missing(value):
        return False
    return value >= min_value


def passes_max(value, max_value) -> bool:
    if max_value is None:
        return True
    if is_missing(value):
        return False
    return value <= max_value


def passes_range(value, min_value=None, max_value=None) -> bool:
    return passes_min(value, min_value) and passes_max(value, max_value)


def passes_bool(value, required_value) -> bool:
    if required_value is None:
        return True
    if is_missing(value):
        return False
    return bool(value) == required_value

In [38]:
def apply_hard_filters(row, hard_filters):
    failed_reasons = []

    # geography
    allowed_country_codes = [
        code.strip().upper()
        for code in hard_filters.country_codes
        if isinstance(code, str) and code.strip()
    ]

    if allowed_country_codes:
        address = row.get("address")

        company_country_code = None
        if isinstance(address, dict):
            company_country_code = address.get("country_code")

        if company_country_code is None or pd.isna(company_country_code):
            failed_reasons.append("country_code_missing")
        else:
            company_country_code = str(company_country_code).strip().upper()
            if company_country_code not in allowed_country_codes:
                failed_reasons.append("country_code_not_allowed")

    # bool
    if not passes_bool(row.get("is_public"), hard_filters.is_public):
        failed_reasons.append("is_public_mismatch_or_missing")

    # year_founded
    if not passes_range(
        row.get("year_founded"),
        hard_filters.year_founded_min,
        hard_filters.year_founded_max,
    ):
        failed_reasons.append("year_founded_out_of_range_or_missing")

    # revenue
    if not passes_range(
        row.get("revenue"),
        hard_filters.revenue_min,
        hard_filters.revenue_max,
    ):
        failed_reasons.append("revenue_out_of_range_or_missing")

    # employee_count
    if not passes_range(
        row.get("employee_count"),
        hard_filters.employee_count_min,
        hard_filters.employee_count_max,
    ):
        failed_reasons.append("employee_count_out_of_range_or_missing")

    return {
        "company_country_code": company_country_code,
        "passed_hard_filters": len(failed_reasons) == 0,
        "hard_filter_failed_reasons": failed_reasons,
    }

In [39]:
candidate_ids = fusion_df["doc_id"].tolist()

candidates_df = df_proc.iloc[candidate_ids].copy().reset_index().rename(
    columns={"index": "doc_id"}
)

candidates_df = candidates_df.merge(
    fusion_df,
    on="doc_id",
    how="left",
)

filter_results = candidates_df.apply(
    lambda row: apply_hard_filters(row, plan.hard_filters),
    axis=1,
)

filter_results_df = pd.DataFrame(filter_results.tolist())

filtered_candidates_df = pd.concat(
    [candidates_df, filter_results_df],
    axis=1,
)

In [40]:
passed_df = filtered_candidates_df[
    filtered_candidates_df["passed_hard_filters"]
].copy()

passed_df = passed_df.sort_values(
    "retrieval_score", ascending=False
).reset_index(drop=True)

In [41]:
print(
    filtered_candidates_df[
        [
            "retrieval_score",
            "operational_name",
            "company_country_code",
            "year_founded",
            "revenue",
            "employee_count",
            "is_public",
            "passed_hard_filters",
            "hard_filter_failed_reasons",
        ]
    ].to_string()
)

    retrieval_score              operational_name company_country_code  year_founded       revenue  employee_count  is_public  passed_hard_filters  hard_filter_failed_reasons
0          0.623569                         Folio                   NO        2018.0  1.050523e+07            24.0      False                 True                          []
1          0.593731                  European Pay                   FR        2022.0  5.136570e+05             2.0      False                 True                          []
2          0.549177                      CTC FUND                   GB        2024.0           NaN             NaN      False                 True                          []
3          0.526933  Euro-Rijn Financial Services                   NL        2020.0  1.751595e+07             NaN      False                 True                          []
4          0.431645                 ECrowd Invest                   ES        2014.0           NaN             NaN      False

In [42]:
print(
    passed_df[
        [
            "retrieval_score",
            "operational_name",
            "company_country_code",
            "year_founded",
            "revenue",
            "employee_count",
            "is_public",
        ]
    ].to_string()
)

    retrieval_score              operational_name company_country_code  year_founded       revenue  employee_count  is_public
0          0.623569                         Folio                   NO        2018.0  1.050523e+07            24.0      False
1          0.593731                  European Pay                   FR        2022.0  5.136570e+05             2.0      False
2          0.549177                      CTC FUND                   GB        2024.0           NaN             NaN      False
3          0.526933  Euro-Rijn Financial Services                   NL        2020.0  1.751595e+07             NaN      False
4          0.431645                 ECrowd Invest                   ES        2014.0           NaN             NaN      False
5          0.240592                          Phin                   DE        2021.0  6.105946e+06             NaN      False
6          0.240269                Rantum Capital                   DE        2013.0  4.688320e+06            19.0    

# Pasul 4

### Normalizarea textelor din companii

In [43]:
def normalize_text(text):
    if text is None:
        return ""
    try:
        if pd.isna(text):
            return ""
    except Exception:
        pass

    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [44]:
def list_field_to_text(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return normalize_text(" ".join(str(x) for x in value if x is not None))
    return normalize_text(str(value))

In [45]:
def naics_entry_to_text(value):
    if value is None:
        return ""

    if isinstance(value, dict):
        parts = []

        label = value.get("label")
        code = value.get("code")

        if label is not None:
            parts.append(str(label))
        if code is not None:
            parts.append(str(code))

        return normalize_text(" ".join(parts))

    return normalize_text(str(value))

In [46]:
def get_naics_text(row):
    primary_text = naics_entry_to_text(row.get("primary_naics"))
    secondary_text = naics_entry_to_text(row.get("secondary_naics"))

    return normalize_text(f"{primary_text} {secondary_text}")

In [47]:
def get_description_text(row):
    return normalize_text(row.get("description"))

In [48]:
def get_business_model_text(row):
    return list_field_to_text(row.get("business_model"))

In [49]:
def get_offerings_text(row):
    return list_field_to_text(row.get("core_offerings"))

In [50]:
def get_target_markets_text(row):
    return list_field_to_text(row.get("target_markets"))

In [51]:
for _, row in passed_df.head(3).iterrows():
    print("NAME:", row["operational_name"])
    print("NAICS:", get_naics_text(row))
    print("DESCRIPTION:", get_description_text(row))
    print("BUSINESS MODEL:", get_business_model_text(row))
    print("OFFERINGS:", get_offerings_text(row))
    print("TARGET MARKETS:", get_target_markets_text(row))
    print("-" * 80)

NAME: Folio
NAICS: financial transactions processing, reserve, and clearinghouse activities 522320
DESCRIPTION: folio as, dba folio, is a norwegian company that provides software-based banking and financial services, including business account management, accounting tools, payroll services, and a business card service. the company serves small businesses and self-employed individuals primarily in norway, with its systems built on top of sparebanken vest's banking infrastructure. folio also offers a mobile application for employees and supports multiple languages.
BUSINESS MODEL: subscription-based business-to-business business-to-consumer
OFFERINGS: payroll services mobile banking application services accounting tools services multilingual financial services financial data integration services business account management services business banking services business card services subscription-based financial services
TARGET MARKETS: financial services
------------------------------------

### Matching simplu pe termeni

In [52]:
def compute_term_match_score(terms, text):
    text = normalize_text(text)

    clean_terms = []
    seen = set()

    for term in terms:
        if term is None:
            continue

        term = normalize_text(term)
        if not term:
            continue

        if term not in seen:
            seen.add(term)
            clean_terms.append(term)

    if not clean_terms:
        return 0.0

    matched = 0
    for term in clean_terms:
        if term in text:
            matched += 1

    return matched / len(clean_terms)

In [53]:
row = passed_df.iloc[0]

industry_terms = plan.industry_terms + plan.naics_terms + plan.canonical_terms

print("NAME:", row["operational_name"])
print("NAICS TEXT:", get_naics_text(row))
print("DESCRIPTION TEXT:", get_description_text(row))
print("INDUSTRY TERMS:", industry_terms)

print("score on naics:", compute_term_match_score(industry_terms, get_naics_text(row)))
print("score on description:", compute_term_match_score(industry_terms, get_description_text(row)))

NAME: Folio
NAICS TEXT: financial transactions processing, reserve, and clearinghouse activities 522320
DESCRIPTION TEXT: folio as, dba folio, is a norwegian company that provides software-based banking and financial services, including business account management, accounting tools, payroll services, and a business card service. the company serves small businesses and self-employed individuals primarily in norway, with its systems built on top of sparebanken vest's banking infrastructure. folio also offers a mobile application for employees and supports multiple languages.
INDUSTRY TERMS: ['fintech', 'financial technology', 'banking', 'financial services', 'Financial Technology', 'Commercial Banking', 'Investment Banking and Securities Dealing', 'Credit Card Issuing', 'Other Nondepository Credit Intermediation', 'Depository Credit Intermediation', 'Nondepository Credit Intermediation', 'fintech', 'financial technology']
score on naics: 0.0
score on description: 0.2


In [54]:
def compute_industry_score(row, plan):
    terms = plan.industry_terms + plan.naics_terms + plan.canonical_terms

    naics_text = get_naics_text(row)
    description_text = get_description_text(row)

    score_naics = compute_term_match_score(terms, naics_text)
    score_description = compute_term_match_score(terms, description_text)

    return 0.6 * score_naics + 0.4 * score_description

In [55]:
def compute_business_model_score(row, plan):
    business_model_text = get_business_model_text(row)
    description_text = get_description_text(row)

    score_bm = compute_term_match_score(plan.business_model_terms, business_model_text)
    score_description = compute_term_match_score(plan.business_model_terms, description_text)

    return 0.7 * score_bm + 0.3 * score_description

In [56]:
def compute_offering_score(row, plan):
    offerings_text = get_offerings_text(row)
    description_text = get_description_text(row)

    score_offerings = compute_term_match_score(plan.offering_terms, offerings_text)
    score_description = compute_term_match_score(plan.offering_terms, description_text)

    return 0.7 * score_offerings + 0.3 * score_description

In [57]:
def compute_target_market_score(row, plan):
    target_markets_text = get_target_markets_text(row)
    description_text = get_description_text(row)

    score_target_markets = compute_term_match_score(plan.target_market_terms, target_markets_text)
    score_description = compute_term_match_score(plan.target_market_terms, description_text)

    return 0.7 * score_target_markets + 0.3 * score_description

In [58]:
def compute_exclude_penalty(row, plan):
    full_text = " ".join([
        get_naics_text(row),
        get_description_text(row),
        get_business_model_text(row),
        get_offerings_text(row),
        get_target_markets_text(row),
    ])

    return compute_term_match_score(plan.exclude_terms, full_text)

In [59]:
def score_candidate(row, plan):
    retrieval_score = row.get("retrieval_score", 0.0)

    try:
        if pd.isna(retrieval_score):
            retrieval_score = 0.0
    except Exception:
        pass

    industry_score = compute_industry_score(row, plan)
    business_model_score = compute_business_model_score(row, plan)
    offering_score = compute_offering_score(row, plan)
    target_market_score = compute_target_market_score(row, plan)
    exclude_penalty = compute_exclude_penalty(row, plan)

    final_score = (
        0.40 * retrieval_score +
        0.20 * industry_score +
        0.15 * business_model_score +
        0.15 * offering_score +
        0.10 * target_market_score -
        0.20 * exclude_penalty
    )

    return {
        "industry_score": industry_score,
        "business_model_score": business_model_score,
        "offering_score": offering_score,
        "target_market_score": target_market_score,
        "exclude_penalty": exclude_penalty,
        "final_score": final_score,
    }

In [60]:
score_results = passed_df.apply(
    lambda row: score_candidate(row, plan),
    axis=1,
)

score_results_df = pd.DataFrame(score_results.tolist())

ranked_df = pd.concat([passed_df, score_results_df], axis=1)

ranked_df = ranked_df.sort_values("final_score", ascending=False).reset_index(drop=True)

In [61]:
print(
    ranked_df[
        [
            "final_score",
            "retrieval_score",
            "industry_score",
            "business_model_score",
            "offering_score",
            "target_market_score",
            "exclude_penalty",
            "operational_name",
            "primary_naics",
            "secondary_naics",
            "business_model",
            "target_markets",
            "core_offerings",
            "description"
        ]
    ].to_string()
)

    final_score  retrieval_score  industry_score  business_model_score  offering_score  target_market_score  exclude_penalty              operational_name                                                                                                                         primary_naics                                                                                  secondary_naics                                                                                     business_model                                                                                                                                                                                target_markets                                                                                                                                                                                                                                                                                                                                      